# 08 — DAP + Few-Shot Fine-Tuning (Arm 2)

**Identical training recipe and evaluation to Notebook 05** — the *only* difference is that
each run starts from the matching **domain-adaptively-pretrained encoder** (`models/dap_<ds>/`
from Notebook 07) instead of the plain CoNLL baseline encoder. That isolates exactly one
variable, so NB05 (Arm 1) vs NB08 (Arm 2) is a clean, budget-matched comparison.

> **Keep this recipe in sync with Notebook 05.** lr `2e-5`, 8 epochs, batch 8, early stopping
> (patience 2), best checkpoint by validation typed-F1. If you change one, mirror it in the
> other before running either.

Reuses the same tokenized inputs NB03 produced (`tokenized/fewshot/...`, `tokenized/<ds>/...`)
and reports **typed F1, boundary F1, and per-entity-type F1** — the same metrics as NB05 plus
the per-type breakdown NB10 needs. Outputs go to `results/dap_fewshot/`.

In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "seqeval==1.2.2" "accelerate>=0.26.0" pandas matplotlib
import torch
print('GPU available:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU available: True


## Step 1 — Mount Drive and configure

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

# --- executed-pipeline layout (identical to Notebooks 03-06) ---
PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
TOKENIZED_DIR  = PROCESSED / 'tokenized'         # HF datasets saved by NB03
LABELS_DIR     = PROCESSED / 'label_maps'        # <ds>_label_map.json (NB03)
MODELS_DIR     = PROCESSED / 'models'            # baseline_conll2003 (NB04)
RESULTS_DIR    = PROCESSED / 'results'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'    # raw jsonl demos (NB02)
BASELINE_DIR   = MODELS_DIR / 'baseline_conll2003'

# shared experiment grid (identical to NB05/NB06)
TARGET_DATASETS = ['wnut17', 'scierc']
BUDGETS = [50, 100, 200]
SEEDS   = [13, 42, 101]

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_label_map(dataset_name):
    with open(LABELS_DIR / f'{dataset_name}_label_map.json') as f:
        m = json.load(f)
    label2id = {str(k): int(v) for k, v in m['label2id'].items()}
    id2label = {int(k): str(v) for k, v in m['id2label'].items()}
    return label2id, id2label

print('processed dir :', PROCESSED)
print('baseline dir  :', BASELINE_DIR, '(exists:', BASELINE_DIR.exists(), ')')

Mounted at /content/drive
processed dir : /content/drive/MyDrive/AAI590/data/processed
baseline dir  : /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003 (exists: True )


In [3]:
import gc, random
import numpy as np
import pandas as pd
from datasets import load_from_disk
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from transformers import (AutoConfig, AutoModelForMaskedLM, AutoModelForTokenClassification,
                          AutoTokenizer, DataCollatorForTokenClassification,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

# ---- recipe: MUST match Notebook 05 ----
LEARNING_RATE = 2e-5
NUM_EPOCHS = 8
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2
SAVE_ALL_MODELS = False

DAP_FEWSHOT_RESULTS_DIR = RESULTS_DIR / 'dap_fewshot'
DAP_FEWSHOT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for ds in TARGET_DATASETS:
    assert (MODELS_DIR / f'dap_{ds}').exists(), f'Run Notebook 07 first (models/dap_{ds} missing).'

## Step 2 — Cache each DAP-adapted encoder

We transplant the DAP encoder into a fresh `bert-base-cased` token-classification model with a
newly-initialized head sized to the target label set — exactly NB05's construction, but the
encoder source is the DAP checkpoint instead of the CoNLL baseline.

In [4]:
dap_encoder_state = {}
for ds_name in TARGET_DATASETS:
    mlm = AutoModelForMaskedLM.from_pretrained(str(MODELS_DIR / f'dap_{ds_name}'))
    dap_encoder_state[ds_name] = {k: v.clone() for k, v in mlm.base_model.state_dict().items()}
    del mlm
print('cached DAP encoders for:', list(dap_encoder_state))

tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_DIR))
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

cached DAP encoders for: ['wnut17', 'scierc']


## Step 3 — Helper functions (identical metrics to Notebook 05)

In [5]:
def collapse_to_boundary(tag):
    if tag == 'O':
        return 'O'
    if tag.startswith('B-'):
        return 'B-ENT'
    if tag.startswith('I-'):
        return 'I-ENT'
    raise ValueError(f'Unexpected BIO tag: {tag}')

def decode_predictions(eval_pred, id2label):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_tags, pred_tags = [], []
    for pred_row, label_row in zip(predictions, labels):
        st, sp = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            st.append(id2label[int(l)]); sp.append(id2label[int(p)])
        true_tags.append(st); pred_tags.append(sp)
    return true_tags, pred_tags

def build_compute_metrics(id2label):
    def compute_metrics(eval_pred):
        true_tags, pred_tags = decode_predictions(eval_pred, id2label)
        tb = [[collapse_to_boundary(t) for t in s] for s in true_tags]
        pb = [[collapse_to_boundary(t) for t in s] for s in pred_tags]
        return {
            'typed_precision': precision_score(true_tags, pred_tags),
            'typed_recall':    recall_score(true_tags, pred_tags),
            'typed_f1':        f1_score(true_tags, pred_tags),
            'boundary_precision': precision_score(tb, pb),
            'boundary_recall':    recall_score(tb, pb),
            'boundary_f1':        f1_score(tb, pb),
        }
    return compute_metrics

def per_type_f1(trainer, dataset, id2label):
    preds, labels, _ = trainer.predict(dataset)
    preds = np.argmax(preds, axis=2)
    true_tags, pred_tags = [], []
    for pr, lr in zip(preds, labels):
        st, sp = [], []
        for p, l in zip(pr, lr):
            if l == -100:
                continue
            st.append(id2label[int(l)]); sp.append(id2label[int(p)])
        true_tags.append(st); pred_tags.append(sp)
    rep = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    out = {}
    for k, v in rep.items():
        if k in ('micro avg', 'macro avg', 'weighted avg'):
            continue
        out[k] = {'precision': float(v['precision']), 'recall': float(v['recall']),
                  'f1': float(v['f1-score']), 'support': int(v['support'])}
    return out

def load_target_datasets(dataset_name, budget, seed):
    fewshot = TOKENIZED_DIR / 'fewshot' / dataset_name / f'budget{budget}_seed{seed}'
    val = TOKENIZED_DIR / dataset_name / 'validation'
    test = TOKENIZED_DIR / dataset_name / 'test'
    for p in (fewshot, val, test):
        assert p.exists(), f'Missing tokenized dataset: {p} (re-run Notebook 03).'
    return load_from_disk(str(fewshot)), load_from_disk(str(val)), load_from_disk(str(test))

def create_target_model(dataset_name):
    label2id, id2label = load_label_map(dataset_name)
    config = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(id2label),
                                        label2id=label2id, id2label=id2label)
    model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', config=config,
                                                            ignore_mismatched_sizes=True)
    # transplant the DAP-adapted encoder (the ONE difference from NB05)
    model.base_model.load_state_dict(dap_encoder_state[dataset_name], strict=True)
    return model, label2id, id2label

def cleanup(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Step 4 — Run one DAP+few-shot experiment

In [6]:
import time

def run_experiment(dataset_name, budget, seed):
    print('=' * 80)
    print(f'[DAP+FS] Dataset={dataset_name} | Budget={budget} | Seed={seed}')
    print('=' * 80)
    set_seed(seed); random.seed(seed); np.random.seed(seed)

    train_ds, val_ds, test_ds = load_target_datasets(dataset_name, budget, seed)
    model, label2id, id2label = create_target_model(dataset_name)

    args = TrainingArguments(
        output_dir=f'/content/dapfs_checkpoints/{dataset_name}_b{budget}_s{seed}',
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='epoch',
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS, weight_decay=WEIGHT_DECAY,
        load_best_model_at_end=True, metric_for_best_model='typed_f1',
        greater_is_better=True, save_total_limit=1, report_to='none',
        seed=seed, data_seed=seed,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=build_compute_metrics(id2label),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    train_output = trainer.train()
    train_seconds = time.time() - t0
    test_metrics = trainer.evaluate(test_ds, metric_key_prefix='test')
    per_type = per_type_f1(trainer, test_ds, id2label)

    if SAVE_ALL_MODELS:
        d = MODELS_DIR / 'dap_fewshot' / dataset_name / f'budget{budget}_seed{seed}'
        d.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(d)); tokenizer.save_pretrained(str(d))

    result = {
        'method': 'dap_fewshot', 'dataset': dataset_name, 'budget': budget, 'seed': seed,
        'num_train_sentences': len(train_ds),
        'training_loss': train_output.training_loss,
        'train_seconds': round(train_seconds, 1),
        'test_typed_precision': test_metrics.get('test_typed_precision'),
        'test_typed_recall':    test_metrics.get('test_typed_recall'),
        'test_typed_f1':        test_metrics.get('test_typed_f1'),
        'test_boundary_precision': test_metrics.get('test_boundary_precision'),
        'test_boundary_recall':    test_metrics.get('test_boundary_recall'),
        'test_boundary_f1':        test_metrics.get('test_boundary_f1'),
        'per_type': json.dumps(per_type),
    }
    print(f"  Typed F1: {result['test_typed_f1']:.4f} | Boundary F1: {result['test_boundary_f1']:.4f}")
    cleanup(trainer, model, train_ds, val_ds, test_ds)
    return result

## Step 5 — Run the full grid (18 runs, resumable)

In [7]:
RESULTS_CSV = DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_results.csv'
RESULTS_JSON = DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_results.json'

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    completed = {(r.dataset, int(r.budget), int(r.seed)) for r in results_df.itertuples()}
    all_results = results_df.to_dict('records')
    print(f'Resuming from {len(all_results)} completed runs.')
else:
    completed, all_results = set(), []

for ds_name in TARGET_DATASETS:
    for budget in BUDGETS:
        for seed in SEEDS:
            if (ds_name, budget, seed) in completed:
                print('Skipping', (ds_name, budget, seed)); continue
            all_results.append(run_experiment(ds_name, budget, seed))
            completed.add((ds_name, budget, seed))
            results_df = pd.DataFrame(all_results).sort_values(['dataset', 'budget', 'seed'])
            results_df.to_csv(RESULTS_CSV, index=False)
            with open(RESULTS_JSON, 'w') as f:
                json.dump(results_df.replace({np.nan: None}).to_dict('records'), f, indent=2, default=str)

results_df = pd.DataFrame(all_results).sort_values(['dataset', 'budget', 'seed']).reset_index(drop=True)
display(results_df[['dataset', 'budget', 'seed', 'test_typed_f1', 'test_boundary_f1']].round(4))
print('Saved ->', RESULTS_CSV)

[DAP+FS] Dataset=wnut17 | Budget=50 | Seed=13


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Typed Precision,Typed Recall,Typed F1,Boundary Precision,Boundary Recall,Boundary F1
1,1.111600,0.863152,0.053333,0.059186,0.056108,0.373626,0.293465,0.328729
2,0.713300,0.787817,0.144086,0.165228,0.153935,0.481947,0.378545,0.424033
3,0.566600,0.688077,0.181889,0.287300,0.222753,0.611765,0.641184,0.626129
4,0.462500,0.659625,0.231169,0.329223,0.271617,0.632850,0.646116,0.639414
5,0.369500,0.655836,0.275926,0.367448,0.315177,0.611442,0.632552,0.621818
6,0.315000,0.670358,0.308880,0.394575,0.346508,0.622412,0.630086,0.626225
7,0.259300,0.660405,0.319885,0.410604,0.359611,0.623798,0.639951,0.631771
8,0.243600,0.662028,0.332689,0.424168,0.372900,0.628606,0.644883,0.636640


early stopping required metric_for_best_model, but did not find eval_typed_f1 so early stopping is disabled


  Typed F1: 0.3614 | Boundary F1: 0.6419


,dataset,budget,seed,test_typed_f1,test_boundary_f1
0,scierc,50,13,0.1194,0.3684
1,scierc,50,42,0.1235,0.3821
2,scierc,50,101,0.0827,0.3895
3,scierc,100,13,0.2481,0.5211
4,scierc,100,42,0.2299,0.5645
5,scierc,100,101,0.1863,0.5190
6,scierc,200,13,0.3879,0.6396
7,scierc,200,42,0.3355,0.6226
8,scierc,200,101,0.3614,0.6419
9,wnut17,50,13,0.2342,0.3070


Saved -> /content/drive/MyDrive/AAI590/data/processed/results/dap_fewshot/dap_fewshot_results.csv


## Step 6 — Aggregate across seeds + compare Arm 1 vs Arm 2

In [8]:
metric_cols = ['test_typed_f1', 'test_boundary_f1']
summary = (results_df.groupby(['dataset', 'budget'])[metric_cols]
           .agg(['mean', 'std']).reset_index())
summary.columns = ['_'.join(str(p) for p in c if str(p)) if isinstance(c, tuple) else c
                   for c in summary.columns]
summary.to_csv(DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_summary_by_budget.csv', index=False)
display(summary.round(4))

# side-by-side with Notebook 05 (Arm 1), if present
ARM1_CSV = RESULTS_DIR / 'fewshot' / 'fewshot_results.csv'
if ARM1_CSV.exists():
    a1 = pd.read_csv(ARM1_CSV)
    a1g = a1.groupby(['dataset', 'budget'])['test_typed_f1'].mean().reset_index()
    a1g = a1g.rename(columns={'test_typed_f1': 'arm1_typed_f1'})
    a2g = results_df.groupby(['dataset', 'budget'])['test_typed_f1'].mean().reset_index()
    a2g = a2g.rename(columns={'test_typed_f1': 'arm2_typed_f1'})
    cmp = a1g.merge(a2g, on=['dataset', 'budget'])
    cmp['dap_gain'] = cmp['arm2_typed_f1'] - cmp['arm1_typed_f1']
    cmp.to_csv(DAP_FEWSHOT_RESULTS_DIR / 'arm1_vs_arm2_typed_f1.csv', index=False)
    print('\nArm 1 (few-shot) vs Arm 2 (DAP + few-shot) -- typed F1, mean over seeds:')
    display(cmp.round(4))
else:
    print('Notebook 05 results not found -- run NB05 for the Arm 1 vs Arm 2 comparison.')

,dataset,budget,test_typed_f1_mean,test_typed_f1_std,test_boundary_f1_mean,test_boundary_f1_std
0,scierc,50,0.1085,0.0224,0.3800,0.0107
1,scierc,100,0.2214,0.0318,0.5348,0.0257
2,scierc,200,0.3616,0.0262,0.6347,0.0106
3,wnut17,50,0.1549,0.1342,0.2137,0.1856
4,wnut17,100,0.3284,0.0067,0.5258,0.0036
5,wnut17,200,0.3823,0.0087,0.5908,0.0180



Arm 1 (few-shot) vs Arm 2 (DAP + few-shot) -- typed F1, mean over seeds:


,dataset,budget,arm1_typed_f1,arm2_typed_f1,dap_gain
0,scierc,50,0.1490,0.1085,-0.0405
1,scierc,100,0.2447,0.2214,-0.0233
2,scierc,200,0.3732,0.3616,-0.0116
3,wnut17,50,0.2971,0.1549,-0.1422
4,wnut17,100,0.3486,0.3284,-0.0202
5,wnut17,200,0.4027,0.3823,-0.0205
